In [ ]:
import os
from os import path
import pickle
import numpy as np
import shutil
from tqdm import tqdm
from glob import glob
import joblib
from scipy.spatial.distance import cdist

from macaquethings.plotting.default_styles import *
from macaquethings.data_util.load_data import get_channel_masks
from macaquethings.plotting.anatomical import plot_data_on_anatomy

import matplotlib.pyplot as plt
import seaborn as sns

figure_style(font_size=6)  # set consistent plotting defaults for all figs

In [ ]:
inter_area_dir = path.join('..', '..', 'results', 'inter_area')

monkey = 'monkeyN'

if monkey == 'monkeyF':
    
    run_names = [
        'monkeyF_v4_target_array_9_stride_2_ridgecv_lfp_threefold',
        'monkeyF_v4_target_array_10_stride_2_ridgecv_lfp_threefold',
        'monkeyF_v4_target_array_11_stride_2_ridgecv_lfp_threefold',
        'monkeyF_v4_target_array_12_stride_2_ridgecv_lfp_threefold',
        'monkeyF_v4_target_array_13_stride_2_ridgecv_lfp_threefold',
    ]
    
    avg_title = r'monkeyF V4 $\to$ IT avg.'
    avg_fname = 'monkeyF_v4_it_avg.svg'
    
    # reference model
    reftimes = [74, 120]
    refdelays = [28, 40]

else:

    run_names = [
        'monkeyN_v4_target_array_13_stride_2_ridgecv_lfp_threefold',
        'monkeyN_v4_target_array_14_stride_2_ridgecv_lfp_threefold',
        'monkeyN_v4_target_array_15_stride_2_ridgecv_lfp_threefold',
        'monkeyN_v4_target_array_16_stride_2_ridgecv_lfp_threefold',
    ]

    avg_title = r'monkeyN V4 $\to$ IT avg.'
    avg_fname = 'monkeyN_v4_it_avg.svg'
    
    # reference model
    reftimes = [62, 112]
    refdelays = [24, 36]
   

savedir = 'weight_matrix_panels'
os.makedirs(savedir, exist_ok=True)

In [ ]:
def load_models(runpath):
    weights, ts, ds = [], [], []
    
    # get paths to all joblib files
    jobfiles = glob(f"{runpath}/**/*.joblib")
    for jobfile in tqdm(jobfiles):
        jobfilename = jobfile.split('/')[-1].split('.')[0]
        t = int(jobfilename.split('_')[2])
        d = int(jobfilename.split('_')[4])
        with open(jobfile, 'rb') as f:
            res = joblib.load(f)
        models = res['models']
        recurrent = models['recurrent']
        inter = models['inter_area']
        ws = []
        for i in range(len(inter)):
            n_target_chans = inter[i].coef_.shape[0]
            n_source_chans = inter[i].coef_.shape[1] - n_target_chans
            W_inter = inter[i].coef_[:,:n_source_chans]
            ws.append(W_inter)
        ws = np.array(ws)
        weights.append(ws)
        ts.append(t)
        ds.append(d)
    return weights, ts, ds 


def compute_dists(run_name):
    weights, ts, ds = load_models(path.join(inter_area_dir, run_name))
    
    weights = np.array(weights)
    weights = np.moveaxis(weights, 1, 0)
    weights = weights.reshape((weights.shape[0], weights.shape[1], -1))
    
    ts = np.array(ts)
    ds = np.array(ds)
    
    indices = []
    for t, d in zip(reftimes, refdelays):
        mask = (ts == t) & (ds == d)
        idx = np.where(mask)[0][0]
        indices.append(idx)
    
    indices = np.array(indices)
    print(indices)

    fold_dists = []
    for i in (range(len(weights))):
        refweights = weights[0, indices]
        dists = cdist(refweights, weights[0], metric='cosine')
        fold_dists.append(dists)
       
    fold_dists = np.array(fold_dists)
    print(fold_dists.shape)

    return ts, ds, fold_dists



fold_dists_per_run = []
for run_name in run_names:
    print(run_name)
    fold_dists_per_run.append(
        compute_dists(run_name)
    )

In [ ]:
fold_dists_per_run[0][2].shape

In [ ]:
# make sure times and delays are sorted the same for all runs

times_per_arr = [r[0] for r in fold_dists_per_run]
delays_per_arr = [r[1] for r in fold_dists_per_run]
dists_per_arr = [r[2] for r in fold_dists_per_run]

times, delays, distances = [], [], []

for ts, ds, dists in zip(times_per_arr, delays_per_arr, dists_per_arr):
    idx = np.lexsort((ds, ts))
    ts = ts[idx]
    ds = ds[idx]
    dists = dists[..., idx]

    times.append(ts)
    delays.append(ds)
    distances.append(dists)

times = np.array(times)
delays = np.array(delays)
distances = np.array(distances)

print(times.shape, delays.shape, distances.shape)

# assert each row has all equal values
t_diffs = times - times[0][None, :]
assert (t_diffs == 0).all()

d_diffs = delays - delays[0][None, :]
assert (d_diffs == 0).all()



# Plot per array

In [ ]:
# plot for each array
cmaps = ['Blues_r', 'Reds_r']
markercolor= 'tab:green'

vmin = distances.min()
vmax = distances.max()

for run_name, ts, ds, dists in zip(run_names, times, delays, distances):
    
    plt.figure(figsize=(THIRD_WIDTH, THIRD_WIDTH))
    print(dists.shape)
    dists = dists.mean(axis=0)
    print(dists[0, :10])
    print(dists.shape)
    for i, (dist, t, d) in enumerate(zip(dists, reftimes, refdelays)):
        plt.subplot(211 + i)
        mappable = plt.tricontourf(ts, ds, dist, levels=100, cmap=cmaps[i], vmin=vmin, vmax=vmax)
        plt.scatter([t], [d], edgecolor='white', s=1, c=markercolor, linewidth=0.2)
        plt.xlabel('time in target array (ms)')
        plt.ylabel('delay (ms)')
        cb = plt.colorbar(shrink=.5, label='cosine dist', mappable=mappable, ticks=np.arange(0, 1.1, .2))
        plt.gca().set_aspect('equal', 'box')
    
        monkey = run_name.split('_')[0]
        source = run_name.split('_')[1]
        target = ' '.join(run_name.split('_')[3:5])
        plt.title(rf"{monkey}  {source} $\to$ {target} (t = {t}ms, {d}ms)")
    plt.savefig(
        path.join(savedir, run_name + '.svg')
    )

# Plot average

In [ ]:
print(dists.shape)
dists = distances.mean(axis=(0, 1))
print(dists[0, :10])
print(dists.shape)

markercolor = 'white'

ts = times[0]
ds = delays[0]

plt.figure(figsize=(THIRD_WIDTH * 1.055, THIRD_WIDTH * 1.055))

for i, (dist, t, d) in enumerate(zip(dists, reftimes, refdelays)):
    plt.subplot(211 + i)
    mappable = plt.tricontourf(ts, ds, dist, levels=100, cmap=cmaps[i], vmin=vmin, vmax=vmax)
    plt.scatter([t], [d], edgecolor='white', s=1, c=markercolor, linewidth=0.2)
    plt.xlabel('time in target array (ms)')
    plt.ylabel('delay (ms)')
    cb = plt.colorbar(shrink=.5, label='cosine dist', mappable=mappable, ticks=np.arange(0, 1.1, .5))
    plt.gca().set_aspect('equal', 'box')

    monkey = run_name.split('_')[0]
    source = run_name.split('_')[1]
    target = ' '.join(run_name.split('_')[3:5])
    plt.title(rf"t = {t}ms, d = {d}ms")
    plt.xticks(np.arange(0,301, 100))
plt.suptitle(avg_title)

plt.savefig(
    path.join(savedir, avg_fname)
)

In [ ]:
savedir